# `parse_pdf` — analyse complète d'un PDF en 1 ouverture fitz

Module testé : [`src/docpipeline/parsing/pdf/parse_pdf.py`](../src/docpipeline/parsing/pdf/parse_pdf.py).

## Ce que fait `parse_pdf(pdf_path)`

Ouvre fitz **une seule fois** et retourne 4 sorties prêtes à être consommées en aval :

| Sortie | Schéma |
|---|---|
| `line_df`  | 1 ligne = 1 ligne de texte (texte, bbox, font, taille, `is_invisible`) |
| `image_df` | 1 ligne = 1 image (bbox affichée en points PDF, dimensions, ratio de couverture) |
| `page_df`  | 1 ligne = 1 page (`page_type`, flags additifs, `extraction_strategy`, `tool`) |
| `doc_summary` | JSON synthèse : `source_tool`, `content_type`, `recommended_strategy`, `pages_needing_ocr` |

**page_type** (mutuellement exclusif) : `native`, `native_with_image`, `scanned`, `scanned_ocr_good`, `scanned_ocr_bad`, `mixed`, `empty`, `unknown`.
**extraction_strategy** : `native` (fitz direct), `ocr` (OCR obligatoire), `hybrid` (texte natif + OCR sur images), `skip` (page vide).

Démo ci-dessous sur un contrat MRH du corpus client (`data/CG contrats MRH/CG Assistance aux personnes AXA.pdf`). Pour le bench complet sur les 71 PDFs du corpus, voir [`05_bench_parse_pdf_js.ipynb`](05_bench_parse_pdf_js.ipynb).

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import json
from docpipeline.parsing.pdf.parse_pdf import parse_pdf

PDF = '../data/CG contrats MRH/CG Assistance aux personnes AXA.pdf'
result = parse_pdf(PDF)

print('=== doc_summary ===')
print(json.dumps(result['doc_summary'], indent=2, ensure_ascii=False, default=str))
print()
print('=== page_df ===')
print(result['page_df'][['page_num', 'page_type', 'tool', 'extraction_strategy', 'char_count', 'n_images', 'image_coverage_ratio']].to_string(index=False))
print()
print(f"line_df  : {len(result['line_df'])} lignes")
print(f"image_df : {len(result['image_df'])} images")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


=== doc_summary ===
{
  "pdf_hash": "4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014",
  "file_size_bytes": 239954,
  "n_pages": 20,
  "source_category": "design_tool",
  "source_tool": "Adobe InDesign",
  "source_confidence": 0.95,
  "source_signals": [
    "creator~adobe indesign",
    "xmp_history~adobe indesign"
  ],
  "creator_raw": "Adobe InDesign 14.0 (Macintosh)",
  "producer_raw": "Adobe PDF Library 15.0",
  "pdf_version": "1.6",
  "creation_date": "D:20210308154957+01'00'",
  "modification_date": "D:20210521105223+02'00'",
  "content_type": "mixed",
  "is_scanned": false,
  "has_text_layer": true,
  "ocr_quality": "not_applicable",
  "ocr_quality_score": null,
  "page_type_counts": {
    "unknown": 1,
    "native": 16,
    "empty": 2,
    "native_with_image": 1
  },
  "scanned_page_ratio": 0.0,
  "native_page_ratio": 0.944,
  "is_encrypted": false,
  "has_form_fields": false,
  "n_vector_tables": 0,
  "n_images_total": 12,
  "recommended_strategy": "per_page_